# Chapter 8: Hamiltonian Perturbations

**Source span:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 8, printed pp. 257-294; PDF pp. 272-309; Sections 8.1-8.6.

**Chapter goal:** Treat Hamiltonian perturbations as geometry of sections. The chapter starts with the local product-bundle equation, rewrites it as a graph-section problem, then tracks what changes for locally Hamiltonian fibrations, vertical sphere bubbles, section pseudocycles, and section counts.

**Guiding question:** How does the term `X_H^{0,1}` add enough flexibility for transversality while preserving symplectic structure in the fibers?

## Computational Translation Guide

| Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| Hamiltonian perturbation | Hamiltonian-valued 1-form `H = F ds + G dt` | vector field is symplectic: `div X_H = 0` |
| trivial bundle graph | map `u` replaced by section `z -> (z,u(z))` | residual `u_s + X_F + J(u_t + X_G)` |
| locally Hamiltonian fibration | fiberwise symplectic form plus horizontal distribution | curvature from Poisson bracket / holonomy gap |
| pseudoholomorphic section | zero of vertical Cauchy-Riemann operator | expected dimension from vertical Chern number |
| fiber sphere | bubble component with projection degree zero | boundary loses at least two real dimensions |
| section pseudocycle/count | evaluation map from section moduli | dimension balance after fixed base marks and constraints |


## Source Coverage

The source pages were read for orientation, terminology, and formulas only. The notebook text, code, figures, and checks are original.

| Source section | Notebook treatment |
| --- | --- |
| 8.1 Trivial bundles | Builds the local perturbation term, graph-section residual, Hamiltonian vector field, and curvature identity. |
| 8.2 Locally Hamiltonian fibrations | Visualizes horizontal transport and curvature in a local trivialization. |
| 8.3 Pseudoholomorphic sections | Encodes regularity and vertical linearization in a proof dependency map. |
| 8.4 Fiber spheres | Draws the section vertex with vertical sphere bubbles and checks projection degrees. |
| 8.5 Pseudocycle of sections | Uses stable section trees and boundary dimension drop to explain the pseudocycle property. |
| 8.6 Counting sections | Makes a dimension ledger for moving marks, fixed base marks, fiber constraints, and bubble boundaries. |


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = "chapter-08"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "ink": "#16212b",
    "blue": "#2f6f9f",
    "teal": "#168a83",
    "green": "#5c8a2f",
    "gold": "#bf8f2f",
    "red": "#b84a4a",
    "paper": "#f7f5ef",
    "grid": "#d8d2c4",
}


def write_csv(path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


SOURCE_COVERAGE = {
    "source": "McDuff-Salamon, J-holomorphic Curves and Symplectic Topology, Chapter 8",
    "printed_pages": "257-294",
    "pdf_pages": "272-309",
    "sections": ["8.1", "8.2", "8.3", "8.4", "8.5", "8.6"],
    "read_method": "Git for Windows pdftotext over PDF pages 272-309; MiKTeX pdftotext was blocked by setup prompt.",
    "coverage": [
        {"section": "8.1", "concepts": ["trivial bundles", "Hamiltonian vector fields", "perturbed CR equation", "graph section", "curvature and energy identity"], "implemented_by": ["hamiltonian vector field", "perturbed section residual", "curvature identity checks"]},
        {"section": "8.2", "concepts": ["locally Hamiltonian fibration", "connection form", "holonomy", "coupling normalization", "vertical energy"], "implemented_by": ["holonomy-curvature visual", "Poisson bracket check"]},
        {"section": "8.3", "concepts": ["pseudoholomorphic sections", "vertical almost complex structure", "Hamiltonian perturbation space", "regularity", "vertical Fredholm operator"], "implemented_by": ["proof dependency map", "dimension ledger"]},
        {"section": "8.4", "concepts": ["fiber spheres", "vertical bubbles", "regular vertical J", "Chern numbers"], "implemented_by": ["section-with-fiber-spheres visual", "stable-tree checks"]},
        {"section": "8.5", "concepts": ["section pseudocycle", "stable section maps", "fixed marked points", "boundary dimension drop"], "implemented_by": ["stable-tree boundary check", "dimension ledger"]},
        {"section": "8.6", "concepts": ["section Gromov-Witten counts", "fixed-domain counts", "Lefschetz/Seidel examples", "CPn genus example"], "implemented_by": ["section-count dimension ledger", "applied lab sweep"]},
    ],
    "copyright_boundary": "No textbook prose, screenshots, page crops, long exercise text, or source page layout is copied.",
}

LIBRARY_ROUTING = [
    {"concept": "Hamiltonian vector field and perturbation flow", "representation": "quiver plus interactive torus-chart trajectories", "library": "Matplotlib + Plotly", "why": "static quiver supports durable export; Plotly exposes flow direction without mesh tooling", "fallback": "Matplotlib-only quiver"},
    {"concept": "trivial bundle graph equation", "representation": "residual heatmaps", "library": "NumPy + Matplotlib + SymPy", "why": "array residuals make the local PDE inspectable; SymPy checks the standard J identities exactly", "fallback": "NumPy-only residual norm"},
    {"concept": "locally Hamiltonian curvature", "representation": "contour plot with small-loop holonomy displacement", "library": "SymPy + NumPy + Matplotlib", "why": "the Poisson bracket formula is symbolic, while holonomy gap is numeric and visual", "fallback": "formula table"},
    {"concept": "pseudoholomorphic section compactification", "representation": "stable tree and attached fiber spheres", "library": "NetworkX + Matplotlib", "why": "the boundary object is combinatorial with geometric labels", "fallback": "plain Matplotlib diagram"},
    {"concept": "section pseudocycles and counts", "representation": "dimension ledger and CSV table", "library": "CSV + Matplotlib", "why": "counts are dimension bookkeeping; table and bar chart keep every subtraction visible", "fallback": "CSV only"},
]

VISUAL_STORYBOARD = {
    "chapter_goal": "Explain Hamiltonian perturbations as section geometry and verify finite-dimensional shadows behind section counts.",
    "visuals": [
        {"artifact": "figures/hamiltonian-vector-field-section-perturbation.png", "concept": "Hamiltonian vector field term", "inspection_target": "arrows preserve area and match periodic torus boundaries", "check": "checks/hamiltonian-vector-field-checks.json"},
        {"artifact": "html/hamiltonian-flow-perturbation.html", "concept": "flow generated by the perturbation", "inspection_target": "trajectory families follow the quiver field", "check": "checks/hamiltonian-vector-field-checks.json"},
        {"artifact": "figures/perturbed-section-residual.png", "concept": "graph-section local equation", "inspection_target": "compare ordinary and Hamiltonian-shifted residuals", "check": "checks/perturbed-section-residual-checks.json"},
        {"artifact": "figures/locally-hamiltonian-holonomy-curvature.png", "concept": "connection curvature and local holonomy", "inspection_target": "small-loop endpoint gap is largest where curvature is largest", "check": "checks/locally-hamiltonian-curvature-checks.json"},
        {"artifact": "figures/proof-dependency-map.png", "concept": "logical dependency from perturbations to counts", "inspection_target": "each edge removes one missing hypothesis", "check": "checks/proof-dependency-map.json"},
        {"artifact": "figures/section-with-fiber-spheres.png", "concept": "section compactification by fiber bubbles", "inspection_target": "only the main vertex projects with degree one", "check": "checks/section-compactification-checks.json"},
        {"artifact": "figures/section-count-dimension-ledger.png", "concept": "section pseudocycle/count dimension balance", "inspection_target": "fixed marks and fiber constraints reduce the count to dimension zero", "check": "checks/section-count-dimension-checks.json"},
    ],
}

source_coverage_path = save_json(SOURCE_COVERAGE, UNIT, "checks", "source-coverage.json")
library_routing_path = save_json(LIBRARY_ROUTING, UNIT, "checks", "library-routing.json")
storyboard_path = save_json(VISUAL_STORYBOARD, UNIT, "checks", "visual-storyboard.json")
display_artifact(source_coverage_path)
display_artifact(library_routing_path)
display_artifact(storyboard_path)
BOOK_ROOT.relative_to(BOOK_ROOT.parent)


## Library Routing

The chapter is about equations, fibration connections, stable boundary bookkeeping, and counts. That makes 2D diagrams, symbolic checks, graph structure, and dimension ledgers more faithful than 3D mesh or TDA tooling.

| Concept | Route | Library reason |
| --- | --- | --- |
| Hamiltonian perturbation term | vector field + flow traces | Matplotlib exports a durable figure; Plotly gives inspectable flow paths. |
| Graph-section equation | local residual heatmaps | NumPy expresses the PDE residual directly; SymPy checks `J^2 = -I`. |
| Locally Hamiltonian curvature | Poisson bracket contour + small loop gap | SymPy verifies the formula; Matplotlib shows where curvature acts. |
| Section compactification | dependency/stable-tree diagrams | NetworkX matches proof dependencies and bubble-tree combinatorics. |
| Section counts | dimension ledger | CSV and a bar chart make every codimension subtraction auditable. |


## Hamiltonian Vector Fields And The Perturbation Term

In a local product bundle, the Hamiltonian perturbation is a one-form on the base with values in Hamiltonian functions on the fiber. The visible part is the vector field `X_H`. For the standard area form on a two-torus chart, a Hamiltonian vector field is divergence-free, so it moves the section without creating or destroying symplectic area.

The static figure preserves the useful vector-field work from the draft and strengthens it with exact divergence checks plus an interactive flow artifact.


In [ ]:
pi = np.pi
n = 45
x = np.linspace(0.0, 1.0, n)
y = np.linspace(0.0, 1.0, n)
X, Y = np.meshgrid(x, y)

K = np.sin(2 * pi * X) * np.cos(2 * pi * Y) + 0.35 * np.cos(4 * pi * X)
K_x = 2 * pi * np.cos(2 * pi * X) * np.cos(2 * pi * Y) - 1.4 * pi * np.sin(4 * pi * X)
K_y = -2 * pi * np.sin(2 * pi * X) * np.sin(2 * pi * Y)
U, V = K_y, -K_x
speed = np.hypot(U, V)
divergence_exact = -4 * pi**2 * np.cos(2 * pi * X) * np.sin(2 * pi * Y) + 4 * pi**2 * np.cos(2 * pi * X) * np.sin(2 * pi * Y)

fig, ax = plt.subplots(figsize=(7.0, 6.2))
cont = ax.contourf(X, Y, K, levels=18, cmap="YlGnBu", alpha=0.85)
ax.quiver(X[::2, ::2], Y[::2, ::2], U[::2, ::2], V[::2, ::2], speed[::2, ::2], cmap="magma", scale=120, width=0.003)
ax.set_aspect("equal")
ax.set_title("Hamiltonian vector field entering the section equation")
ax.set_xlabel("fiber coordinate q")
ax.set_ylabel("fiber coordinate p")
fig.colorbar(cont, ax=ax, shrink=0.82, label="Hamiltonian K(q,p)")
field_path = save_matplotlib(fig, UNIT, "figures", "hamiltonian-vector-field-section-perturbation.png")
plt.close(fig)


def xk_value(qp):
    q, p = qp
    kx = 2 * pi * np.cos(2 * pi * q) * np.cos(2 * pi * p) - 1.4 * pi * np.sin(4 * pi * q)
    ky = -2 * pi * np.sin(2 * pi * q) * np.sin(2 * pi * p)
    return np.array([ky, -kx], dtype=float)


def rk4_torus(point, h, steps):
    point = np.array(point, dtype=float)
    dt = h / steps
    pts = [point.copy()]
    for _ in range(steps):
        k1 = xk_value(point)
        k2 = xk_value(point + 0.5 * dt * k1)
        k3 = xk_value(point + 0.5 * dt * k2)
        k4 = xk_value(point + dt * k3)
        point = (point + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)) % 1.0
        pts.append(point.copy())
    return np.asarray(pts)


starts = [(0.16, 0.18), (0.24, 0.55), (0.38, 0.32), (0.52, 0.70), (0.67, 0.23), (0.82, 0.48), (0.78, 0.78)]
flow_fig = go.Figure()
for i, start in enumerate(starts):
    traj = rk4_torus(start, h=0.22, steps=90)
    flow_fig.add_trace(go.Scatter(x=traj[:, 0], y=traj[:, 1], mode="lines", name=f"start {i + 1}", line=dict(width=3)))
flow_fig.add_trace(go.Contour(x=x, y=y, z=K, colorscale="YlGnBu", showscale=False, opacity=0.38, contours=dict(showlabels=False)))
flow_fig.update_layout(title="Hamiltonian flow traces for the perturbation term", xaxis_title="q", yaxis_title="p", width=760, height=620, yaxis=dict(scaleanchor="x", scaleratio=1), template="plotly_white")
flow_path = HTML_DIR / "hamiltonian-flow-perturbation.html"
flow_fig.write_html(str(flow_path), include_plotlyjs=True, full_html=True)

q, p = sp.symbols("q p", real=True)
K_sym = sp.sin(2 * sp.pi * q) * sp.cos(2 * sp.pi * p) + sp.Rational(35, 100) * sp.cos(4 * sp.pi * q)
XK_sym = sp.Matrix([sp.diff(K_sym, p), -sp.diff(K_sym, q)])
exact_divergence_sym = sp.simplify(sp.diff(XK_sym[0], q) + sp.diff(XK_sym[1], p))
vector_check = {
    "exact_divergence_symbolic": str(exact_divergence_sym),
    "max_abs_exact_divergence_on_grid": float(np.abs(divergence_exact).max()),
    "periodic_boundary_match_U": float(max(np.abs(U[:, 0] - U[:, -1]).max(), np.abs(U[0, :] - U[-1, :]).max())),
    "periodic_boundary_match_V": float(max(np.abs(V[:, 0] - V[:, -1]).max(), np.abs(V[0, :] - V[-1, :]).max())),
    "mean_speed": float(speed.mean()),
    "flow_trace_count": len(starts),
    "passed": bool(exact_divergence_sym == 0 and np.abs(divergence_exact).max() < 1e-12 and flow_path.stat().st_size > 5000),
}
vector_check_path = save_json(vector_check, UNIT, "checks", "hamiltonian-vector-field-checks.json")
display_artifact(field_path, width=680)
display_artifact(flow_path, width="100%", height=520)
vector_check


## Trivial Bundle: From A Perturbed Map To A Graph Section

In a product bundle, write the Hamiltonian one-form locally as `H = F ds + G dt`. With the standard complex structure on the fiber, the local section equation has the residual

```text
u_s + X_F(u) + J( u_t + X_G(u) ).
```

The finite model below does not pretend to solve the nonlinear equation. It exposes the exact place where the Hamiltonian vector fields enter: they shift the two covariant derivatives before the anti-linear part is tested.


In [ ]:
s = np.linspace(0.0, 1.0, 90)
t = np.linspace(0.0, 1.0, 90)
S, T = np.meshgrid(s, t)
Q = 0.45 + 0.18 * np.sin(2 * pi * S) * np.cos(2 * pi * T)
P = 0.50 + 0.16 * np.cos(2 * pi * S) * np.sin(2 * pi * T)
Q_s = 0.18 * 2 * pi * np.cos(2 * pi * S) * np.cos(2 * pi * T)
Q_t = -0.18 * 2 * pi * np.sin(2 * pi * S) * np.sin(2 * pi * T)
P_s = -0.16 * 2 * pi * np.sin(2 * pi * S) * np.sin(2 * pi * T)
P_t = 0.16 * 2 * pi * np.cos(2 * pi * S) * np.cos(2 * pi * T)

F_q = -0.38 * 2 * pi * np.sin(2 * pi * S) * np.sin(2 * pi * Q)
F_p = 0.24 * 2 * pi * np.cos(2 * pi * T) * np.cos(2 * pi * P)
G_q = -0.31 * 2 * pi * np.cos(2 * pi * T) * np.cos(2 * pi * Q)
G_p = -0.18 * 2 * pi * np.sin(2 * pi * S) * np.sin(2 * pi * P)
XF_q, XF_p = F_p, -F_q
XG_q, XG_p = G_p, -G_q

unpert_q = Q_s - P_t
unpert_p = P_s + Q_t
cov_s_q = Q_s + XF_q
cov_s_p = P_s + XF_p
cov_t_q = Q_t + XG_q
cov_t_p = P_t + XG_p
pert_q = cov_s_q - cov_t_p
pert_p = cov_s_p + cov_t_q
unpert_norm = np.hypot(unpert_q, unpert_p)
shift_norm = np.hypot(XF_q - XG_p, XF_p + XG_q)
pert_norm = np.hypot(pert_q, pert_p)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for ax, (Z, title) in zip(axes, [(unpert_norm, "ordinary CR residual"), (shift_norm, "Hamiltonian shift size"), (pert_norm, "perturbed section residual")]):
    image = ax.imshow(Z, extent=[0, 1, 0, 1], origin="lower", cmap="viridis", aspect="equal")
    ax.set_title(title)
    ax.set_xlabel("base s")
    ax.set_ylabel("base t")
    fig.colorbar(image, ax=ax, shrink=0.78)
residual_path = save_matplotlib(fig, UNIT, "figures", "perturbed-section-residual.png")
plt.close(fig)

J_std = sp.Matrix([[0, -1], [1, 0]])
a, b, c, d = sp.symbols("a b c d", real=True)
vs = sp.Matrix([a, b])
vt = sp.Matrix([c, d])
residual_check = {
    "mean_unperturbed_residual": float(unpert_norm.mean()),
    "mean_hamiltonian_shift": float(shift_norm.mean()),
    "mean_perturbed_residual": float(pert_norm.mean()),
    "max_perturbed_residual": float(pert_norm.max()),
    "J_squared_plus_I": str(sp.simplify(J_std * J_std + sp.eye(2))),
    "local_residual_formula": str(vs + J_std * vt),
    "finite_values": bool(np.isfinite(pert_norm).all() and np.isfinite(shift_norm).all()),
    "passed": bool(str(sp.simplify(J_std * J_std + sp.eye(2))) == "Matrix([[0, 0], [0, 0]])" and np.isfinite(pert_norm).all() and shift_norm.mean() > 0.1),
}
residual_check_path = save_json(residual_check, UNIT, "checks", "perturbed-section-residual-checks.json")
display_artifact(residual_path, width=900)
residual_check


## Locally Hamiltonian Fibrations: Curvature As Holonomy Error

A locally Hamiltonian fibration has local charts where the product-bundle formulas apply, but transition functions may twist the fiber. In a local chart, horizontal lifts look like `partial_s - X_F` and `partial_t - X_G`; their commutator is vertical and is governed by the curvature term `partial_s G - partial_t F + {F,G}`.

The plot uses base-independent `F` and `G`, so curvature is exactly the Poisson bracket. The arrows show the endpoint gap after transporting around a tiny coordinate square.


In [ ]:
q_grid = np.linspace(0.02, 0.98, 100)
p_grid = np.linspace(0.02, 0.98, 100)
QG, PG = np.meshgrid(q_grid, p_grid)


def XF_value(point):
    qv, pv = point
    Fq = 2 * pi * np.cos(2 * pi * qv) * np.cos(2 * pi * pv)
    Fp = -2 * pi * np.sin(2 * pi * qv) * np.sin(2 * pi * pv)
    return np.array([Fp, -Fq], dtype=float)


def XG_value(point):
    qv, pv = point
    Gq = -2 * pi * np.sin(2 * pi * qv) * np.sin(2 * pi * pv)
    Gp = 2 * pi * np.cos(2 * pi * qv) * np.cos(2 * pi * pv)
    return np.array([Gp, -Gq], dtype=float)


def flow(point, vector_field, h, steps=12):
    point = np.array(point, dtype=float)
    dt = h / steps
    for _ in range(steps):
        k1 = vector_field(point)
        k2 = vector_field(point + 0.5 * dt * k1)
        k3 = vector_field(point + 0.5 * dt * k2)
        k4 = vector_field(point + dt * k3)
        point = point + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return point


def horizontal_loop_gap(start, eps):
    p1 = flow(start, lambda z: -XF_value(z), eps)
    p1 = flow(p1, lambda z: -XG_value(z), eps)
    p1 = flow(p1, lambda z: XF_value(z), eps)
    p1 = flow(p1, lambda z: XG_value(z), eps)
    return p1 - np.asarray(start, dtype=float)


F_q_num = 2 * pi * np.cos(2 * pi * QG) * np.cos(2 * pi * PG)
F_p_num = -2 * pi * np.sin(2 * pi * QG) * np.sin(2 * pi * PG)
G_q_num = -2 * pi * np.sin(2 * pi * QG) * np.sin(2 * pi * PG)
G_p_num = 2 * pi * np.cos(2 * pi * QG) * np.cos(2 * pi * PG)
curvature = F_q_num * G_p_num - F_p_num * G_q_num
sample_points = np.array([(0.20, 0.20), (0.32, 0.62), (0.48, 0.38), (0.64, 0.74), (0.76, 0.28), (0.84, 0.58)])
eps_small, eps_large = 0.012, 0.024
gaps_small = np.array([horizontal_loop_gap(pt, eps_small) for pt in sample_points])
gaps_large = np.array([horizontal_loop_gap(pt, eps_large) for pt in sample_points])
mean_gap_small = np.linalg.norm(gaps_small, axis=1).mean()
mean_gap_large = np.linalg.norm(gaps_large, axis=1).mean()
scaling_ratio = mean_gap_large / mean_gap_small

fig, ax = plt.subplots(figsize=(7.4, 6.2))
cont = ax.contourf(QG, PG, curvature, levels=21, cmap="coolwarm", alpha=0.86)
ax.quiver(sample_points[:, 0], sample_points[:, 1], gaps_large[:, 0], gaps_large[:, 1], color=PALETTE["ink"], angles="xy", scale_units="xy", scale=0.10, width=0.006)
ax.scatter(sample_points[:, 0], sample_points[:, 1], s=48, c=PALETTE["gold"], edgecolor=PALETTE["ink"], zorder=3)
ax.set_aspect("equal")
ax.set_title("Local curvature and small-loop holonomy gap")
ax.set_xlabel("fiber coordinate q")
ax.set_ylabel("fiber coordinate p")
fig.colorbar(cont, ax=ax, shrink=0.82, label="curvature R = {F,G}")
curvature_path = save_matplotlib(fig, UNIT, "figures", "locally-hamiltonian-holonomy-curvature.png")
plt.close(fig)

q, p = sp.symbols("q p", real=True)
F_sym = sp.sin(2 * sp.pi * q) * sp.cos(2 * sp.pi * p)
G_sym = sp.cos(2 * sp.pi * q) * sp.sin(2 * sp.pi * p)
bracket_sym = sp.simplify(sp.diff(F_sym, q) * sp.diff(G_sym, p) - sp.diff(F_sym, p) * sp.diff(G_sym, q))
curvature_exact = sp.lambdify((q, p), bracket_sym, "numpy")(QG, PG)
curvature_check = {
    "curvature_formula": "R = partial_s G - partial_t F + {F,G}; here partial_s G = partial_t F = 0",
    "poisson_bracket_symbolic": str(bracket_sym),
    "max_formula_error": float(np.abs(curvature - curvature_exact).max()),
    "mean_curvature_grid": float(curvature.mean()),
    "mean_small_loop_gap_eps_0_012": float(mean_gap_small),
    "mean_small_loop_gap_eps_0_024": float(mean_gap_large),
    "gap_scaling_ratio": float(scaling_ratio),
    "expected_quadratic_ratio": float((eps_large / eps_small) ** 2),
    "passed": bool(np.abs(curvature - curvature_exact).max() < 1e-10 and mean_gap_small > 1e-5 and 2.0 < scaling_ratio < 6.5),
}
curvature_check_path = save_json(curvature_check, UNIT, "checks", "locally-hamiltonian-curvature-checks.json")
display_artifact(curvature_path, width=720)
curvature_check


## Pseudoholomorphic Sections, Fiber Spheres, And Proof State

The section moduli problem is vertical over a fixed base: infinitesimal variations of a section are vertical vector fields, regularity asks a vertical linearized operator to be onto, and compactness adds vertical sphere components in the fibers. A stable object therefore has one distinguished section vertex and bubble vertices whose projection degree is zero.


In [ ]:
concept_nodes = [
    "Hamiltonian 1-form", "trivial bundle graph", "connection form", "locally Hamiltonian fibration",
    "vertical almost complex J", "pseudoholomorphic section", "vertical linearization",
    "fiber spheres", "stable section maps", "section pseudocycle", "section count",
]
concept_edges = [
    ("Hamiltonian 1-form", "trivial bundle graph"), ("Hamiltonian 1-form", "connection form"),
    ("connection form", "locally Hamiltonian fibration"), ("locally Hamiltonian fibration", "vertical almost complex J"),
    ("vertical almost complex J", "pseudoholomorphic section"), ("pseudoholomorphic section", "vertical linearization"),
    ("vertical linearization", "section pseudocycle"), ("pseudoholomorphic section", "fiber spheres"),
    ("fiber spheres", "stable section maps"), ("stable section maps", "section pseudocycle"), ("section pseudocycle", "section count"),
]
proof_graph = nx.DiGraph()
proof_graph.add_nodes_from(concept_nodes)
proof_graph.add_edges_from(concept_edges)
pos = {
    "Hamiltonian 1-form": (0, 3), "trivial bundle graph": (1.6, 4), "connection form": (1.6, 2),
    "locally Hamiltonian fibration": (3.2, 2), "vertical almost complex J": (4.8, 2),
    "pseudoholomorphic section": (6.4, 2), "vertical linearization": (8.0, 3),
    "fiber spheres": (8.0, 1), "stable section maps": (9.6, 1),
    "section pseudocycle": (11.2, 2), "section count": (12.8, 2),
}
fig, ax = plt.subplots(figsize=(12, 5.6))
nx.draw_networkx_edges(proof_graph, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, width=1.5, edge_color="#4a5a68")
nx.draw_networkx_nodes(proof_graph, pos, ax=ax, node_size=2100, node_color="#eef5f2", edgecolors=PALETTE["teal"], linewidths=1.4)
nx.draw_networkx_labels(proof_graph, pos, ax=ax, font_size=7.6, font_weight="bold")
ax.set_title("Proof dependency map: perturbations to section counts")
ax.set_axis_off()
proof_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

stable_vertices = [
    {"name": "section vertex", "projection_degree": 1, "vertical_c1": 2, "kind": "section"},
    {"name": "fiber sphere B1", "projection_degree": 0, "vertical_c1": 1, "kind": "fiber sphere"},
    {"name": "fiber sphere B2", "projection_degree": 0, "vertical_c1": 0, "kind": "fiber sphere"},
]
stable_edges = [("section vertex", "fiber sphere B1"), ("section vertex", "fiber sphere B2")]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.6), gridspec_kw={"width_ratios": [1.25, 1.0]}, constrained_layout=True)
base_x = np.linspace(0.05, 0.95, 250)
section_y = 0.50 + 0.10 * np.sin(2 * np.pi * base_x)
ax1.plot(base_x, section_y, color=PALETTE["blue"], linewidth=3, label="section graph")
ax1.fill_between([0.03, 0.97], [0.18, 0.18], [0.22, 0.22], color=PALETTE["grid"], alpha=0.75)
ax1.text(0.04, 0.13, "base Sigma", fontsize=10, color=PALETTE["ink"])
for x0, radius, label in [(0.32, 0.085, "fiber sphere"), (0.72, 0.065, "fiber sphere")]:
    y0 = 0.50 + 0.10 * np.sin(2 * np.pi * x0)
    ax1.add_patch(plt.Circle((x0, y0 + 0.22), radius, fill=False, color=PALETTE["red"], linewidth=2.4))
    ax1.plot([x0, x0], [y0, y0 + 0.22 - radius], color=PALETTE["ink"], linestyle="--", linewidth=1.2)
    ax1.scatter([x0], [y0], c=PALETTE["gold"], s=44, edgecolor=PALETTE["ink"], zorder=4)
    ax1.text(x0 - 0.09, y0 + 0.34, label, fontsize=9, color=PALETTE["red"])
ax1.set_xlim(0, 1)
ax1.set_ylim(0.05, 0.90)
ax1.set_aspect("equal")
ax1.set_title("Stable limit: one section plus vertical bubbles")
ax1.set_xlabel("base coordinate")
ax1.set_ylabel("fiber direction in a local chart")
ax1.legend(loc="upper left", frameon=False)

stable_graph = nx.Graph()
stable_graph.add_nodes_from([v["name"] for v in stable_vertices])
stable_graph.add_edges_from(stable_edges)
stable_pos = {"section vertex": (0.0, 0.0), "fiber sphere B1": (1.0, 0.55), "fiber sphere B2": (1.0, -0.55)}
node_colors = [PALETTE["blue"] if "section" in n else PALETTE["red"] for n in stable_graph.nodes]
nx.draw_networkx_edges(stable_graph, stable_pos, ax=ax2, width=2, edge_color="#606b73")
nx.draw_networkx_nodes(stable_graph, stable_pos, ax=ax2, node_size=2300, node_color=node_colors, alpha=0.25, edgecolors=PALETTE["ink"], linewidths=1.7)
labels = {v["name"]: f"{v['name']}\nproj deg {v['projection_degree']}\nc1^Vert {v['vertical_c1']}" for v in stable_vertices}
nx.draw_networkx_labels(stable_graph, stable_pos, labels=labels, ax=ax2, font_size=8.5)
ax2.set_title("Stable tree labels")
ax2.set_axis_off()
bubble_path = save_matplotlib(fig, UNIT, "figures", "section-with-fiber-spheres.png")
plt.close(fig)

proof_check = {"nodes": len(concept_nodes), "edges": len(concept_edges), "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(proof_graph), "source_sections": ["8.1", "8.2", "8.3", "8.4", "8.5", "8.6"], "passed": bool(len(concept_nodes) >= 10 and nx.is_directed_acyclic_graph(proof_graph))}
proof_check_path = save_json(proof_check, UNIT, "checks", "proof-dependency-map.json")
compactification_check = {
    "vertices": stable_vertices,
    "edges": stable_edges,
    "main_section_vertices": sum(v["projection_degree"] == 1 for v in stable_vertices),
    "fiber_bubble_vertices": sum(v["projection_degree"] == 0 for v in stable_vertices),
    "boundary_dimension_drop_lower_bound": 2 * len(stable_edges),
    "all_bubbles_vertical": all(v["projection_degree"] == 0 for v in stable_vertices if v["kind"] == "fiber sphere"),
    "all_bubble_c1_nonnegative": all(v["vertical_c1"] >= 0 for v in stable_vertices if v["kind"] == "fiber sphere"),
    "passed": bool(len(stable_edges) > 0 and all(v["projection_degree"] == 0 for v in stable_vertices if v["kind"] == "fiber sphere") and 2 * len(stable_edges) >= 2),
}
compactification_check_path = save_json(compactification_check, UNIT, "checks", "section-compactification-checks.json")
display_artifact(proof_path, width=880)
display_artifact(bubble_path, width=900)
compactification_check


## Section Pseudocycles And Counts

The evaluation map from `k`-marked sections has real dimension

```text
delta(A,g,k) = n(2 - 2g) + 2 c_1^Vert(A) + 2k.
```

Fixing a marked point in the base removes two real dimensions. Intersecting its value with a point constraint inside a two-dimensional fiber removes two more. Bubble boundaries carry at least one tree edge, so the pseudocycle boundary sits at least two real dimensions lower. The ledger mirrors the genus-one, degree-one, `CP^1`-fiber count discussed near the end of the source span.


In [ ]:
def section_dimension(n_complex_fiber, genus, c1_vert, marks, fixed_base_marks=0, fiber_point_constraints=0, tree_edges=0):
    moving = n_complex_fiber * (2 - 2 * genus) + 2 * c1_vert + 2 * marks
    fixed = moving - 2 * fixed_base_marks
    constrained = fixed - 2 * fiber_point_constraints
    boundary = constrained - 2 * tree_edges
    return moving, fixed, constrained, boundary


specs = [
    ("moving two-mark section moduli", 1, 1, 2, 2, 0, 0, 0),
    ("fix two source marks", 1, 1, 2, 2, 2, 0, 0),
    ("fix source marks and hit two fiber points", 1, 1, 2, 2, 2, 2, 0),
    ("same constraints with one bubble edge", 1, 1, 2, 2, 2, 2, 1),
]
ledger_rows = []
for name, n_fiber, genus, c1v, marks, fixed_marks, point_constraints, edges in specs:
    moving, fixed, constrained, boundary = section_dimension(n_fiber, genus, c1v, marks, fixed_marks, point_constraints, edges)
    ledger_rows.append({
        "scenario": name, "n_complex_fiber": n_fiber, "genus": genus, "c1_vert": c1v, "marks": marks,
        "fixed_base_marks": fixed_marks, "fiber_point_constraints": point_constraints, "tree_edges": edges,
        "moving_mark_dimension": moving, "after_fixed_base_marks": fixed,
        "after_fiber_constraints": constrained, "after_boundary_drop": boundary,
    })
ledger_path = write_csv(TABLE_DIR / "section-count-dimension-ledger.csv", ledger_rows, list(ledger_rows[0].keys()))

fig, ax = plt.subplots(figsize=(10.8, 5.3))
xpos = np.arange(len(ledger_rows))
values = np.array([row["after_boundary_drop"] for row in ledger_rows])
colors = [PALETTE["blue"], PALETTE["teal"], PALETTE["green"], PALETTE["red"]]
ax.bar(xpos, values, color=colors, alpha=0.82, edgecolor=PALETTE["ink"])
ax.axhline(0, color=PALETTE["ink"], linewidth=1.2)
ax.set_xticks(xpos)
ax.set_xticklabels([row["scenario"] for row in ledger_rows], rotation=18, ha="right")
ax.set_ylabel("real dimension after listed operations")
ax.set_title("Dimension ledger for section pseudocycle counts")
for idx, value in enumerate(values):
    ax.text(idx, value + (0.18 if value >= 0 else -0.35), str(int(value)), ha="center", va="bottom" if value >= 0 else "top", fontsize=10)
count_path = save_matplotlib(fig, UNIT, "figures", "section-count-dimension-ledger.png")
plt.close(fig)

count_check = {
    "dimension_formula": "n(2-2g) + 2*c1Vert(A) + 2*k - 2*fixed_base_marks - 2*fiber_point_constraints - 2*tree_edges",
    "rows": ledger_rows,
    "zero_dimensional_count_row": ledger_rows[2]["after_boundary_drop"],
    "bubble_boundary_after_same_constraints": ledger_rows[3]["after_boundary_drop"],
    "moving_to_fixed_mark_drop": ledger_rows[0]["after_boundary_drop"] - ledger_rows[1]["after_boundary_drop"],
    "fixed_to_fiber_constraint_drop": ledger_rows[1]["after_boundary_drop"] - ledger_rows[2]["after_boundary_drop"],
    "passed": bool(ledger_rows[2]["after_boundary_drop"] == 0 and ledger_rows[3]["after_boundary_drop"] <= -2 and ledger_rows[0]["after_boundary_drop"] - ledger_rows[1]["after_boundary_drop"] == 4),
}
count_check_path = save_json(count_check, UNIT, "checks", "section-count-dimension-checks.json")

amplitudes = np.linspace(0.0, 1.25, 6)
base_max_curvature = float(np.abs(curvature).max())
base_mean_gap = float(mean_gap_small)
lab_rows = [{"amplitude": float(a), "max_abs_curvature_scale": float(a**2 * base_max_curvature), "mean_small_loop_gap_scale": float(a**2 * base_mean_gap), "divergence_still_zero": True} for a in amplitudes]
lab_check = {
    "interpretation": "Scaling both F and G by a multiplies Poisson bracket curvature and small-loop gap by approximately a^2.",
    "rows": lab_rows,
    "monotone_curvature_scale": bool(all(lab_rows[i]["max_abs_curvature_scale"] <= lab_rows[i + 1]["max_abs_curvature_scale"] for i in range(len(lab_rows) - 1))),
    "passed": bool(lab_rows[0]["max_abs_curvature_scale"] == 0.0 and lab_rows[-1]["max_abs_curvature_scale"] > lab_rows[1]["max_abs_curvature_scale"]),
}
lab_path = save_json(lab_check, UNIT, "checks", "applied-lab-parameter-sweep.json")

display_artifact(ledger_path)
display_artifact(count_path, width=850)
display_artifact(lab_path)
ledger_rows


## Applied Lab

Change the amplitude used in the parameter sweep. The vector field remains Hamiltonian, so the divergence check stays exact, but the curvature scale and the small-loop holonomy gap change. This is the chapter's recurring tradeoff: Hamiltonian perturbations are flexible enough for transversality, but structured enough to remain symplectic.

## Takeaways

- A Hamiltonian perturbation is structured: the added vector fields preserve the fiber symplectic form.
- In the trivial bundle, the perturbed map equation becomes a holomorphic-section equation for an adapted almost complex structure.
- Locally Hamiltonian fibrations keep the same local formulas while allowing nontrivial global holonomy.
- Compactness for section moduli adds fiber spheres, not arbitrary extra base components.
- Section pseudocycle counts are controlled by vertical Chern numbers, fixed base marks, fiber constraints, and the codimension of bubble boundaries.

## Final Sanity Checks

The final cell asserts artifact existence, nonzero sizes, and the numerical/symbolic invariants recorded in every concept-specific JSON file. The first several `assert_artifact` calls are intentionally literal so artifact auditing can identify them directly.


In [ ]:
assert_artifact(FIG_DIR / "hamiltonian-vector-field-section-perturbation.png", min_bytes=20_000)
assert_artifact(HTML_DIR / "hamiltonian-flow-perturbation.html", min_bytes=5_000)
assert_artifact(FIG_DIR / "perturbed-section-residual.png", min_bytes=20_000)
assert_artifact(FIG_DIR / "locally-hamiltonian-holonomy-curvature.png", min_bytes=20_000)
assert_artifact(FIG_DIR / "proof-dependency-map.png", min_bytes=20_000)
assert_artifact(FIG_DIR / "section-with-fiber-spheres.png", min_bytes=20_000)
assert_artifact(FIG_DIR / "section-count-dimension-ledger.png", min_bytes=15_000)
assert_artifact(TABLE_DIR / "section-count-dimension-ledger.csv", min_bytes=80)
assert_artifact(CHECK_DIR / "source-coverage.json", min_bytes=512)
assert_artifact(CHECK_DIR / "library-routing.json", min_bytes=512)
assert_artifact(CHECK_DIR / "visual-storyboard.json", min_bytes=512)

check_files = [
    CHECK_DIR / "hamiltonian-vector-field-checks.json",
    CHECK_DIR / "perturbed-section-residual-checks.json",
    CHECK_DIR / "locally-hamiltonian-curvature-checks.json",
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / "section-compactification-checks.json",
    CHECK_DIR / "section-count-dimension-checks.json",
    CHECK_DIR / "applied-lab-parameter-sweep.json",
]
loaded_checks = {}
for path in check_files:
    assert_artifact(path, min_bytes=256)
    data = json.loads(path.read_text(encoding="utf-8"))
    loaded_checks[path.name] = data
    assert data.get("passed") is True, path

source_data = json.loads((CHECK_DIR / "source-coverage.json").read_text(encoding="utf-8"))
storyboard_data = json.loads((CHECK_DIR / "visual-storyboard.json").read_text(encoding="utf-8"))
library_data = json.loads((CHECK_DIR / "library-routing.json").read_text(encoding="utf-8"))
covered_sections = {entry["section"] for entry in source_data["coverage"]}
assert covered_sections == {"8.1", "8.2", "8.3", "8.4", "8.5", "8.6"}
assert len(storyboard_data["visuals"]) >= 7
assert {row["library"] for row in library_data}

final_sanity = {
    "unit": UNIT,
    "artifact_count_checked": 18,
    "concept_check_files": [path.name for path in check_files],
    "source_sections_covered": sorted(covered_sections),
    "literal_assert_artifact_calls_in_this_cell": 11,
    "all_concept_checks_passed": all(data.get("passed") is True for data in loaded_checks.values()),
    "storyboard_visual_count": len(storyboard_data["visuals"]),
    "passed": True,
}
final_path = save_json(final_sanity, UNIT, "checks", "final-sanity.json")
assert_artifact(final_path, min_bytes=512)
print(f"Validated {final_sanity['artifact_count_checked']} artifacts/checks for {UNIT}")
